# 🏎️ F1 Driver–Circuit Compatibility
## Notebook 02 — Data Preprocessing

**What this notebook does:**
1. Loads `laps_raw.csv` and `results_raw.csv`
2. Converts timedelta strings to float seconds
3. Removes pit-in / pit-out laps
4. Removes Safety Car / VSC / Red Flag laps
5. Removes laps with missing or impossible lap times
6. Removes outlier laps (>3σ above race median)
7. Drops DNF drivers who completed ≤50% of laps
8. Normalizes lap times and sector times within each race
9. Saves `laps_clean.csv` and `results_clean.csv`

**Input:**  `data/raw/laps_raw.csv`, `data/raw/results_raw.csv`  
**Output:** `data/processed/laps_clean.csv`, `data/processed/results_clean.csv`

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import os
import warnings
warnings.filterwarnings('ignore')

print('✅ Imports ready.')

## Step 1 — Configure Paths

In [ ]:
NOTEBOOK_DIR = os.path.dirname(os.path.abspath('__file__'))
PROJECT_ROOT = os.path.abspath(os.path.join(NOTEBOOK_DIR, '..'))

RAW_DIR  = os.path.join(PROJECT_ROOT, 'data', 'raw')
PROC_DIR = os.path.join(PROJECT_ROOT, 'data', 'processed')
os.makedirs(PROC_DIR, exist_ok=True)

LAPS_RAW_PATH      = os.path.join(RAW_DIR,  'laps_raw.csv')
RESULTS_RAW_PATH   = os.path.join(RAW_DIR,  'results_raw.csv')
LAPS_CLEAN_PATH    = os.path.join(PROC_DIR, 'laps_clean.csv')
RESULTS_CLEAN_PATH = os.path.join(PROC_DIR, 'results_clean.csv')

for path in [LAPS_RAW_PATH, RESULTS_RAW_PATH]:
    exists = os.path.exists(path)
    print(f"  {'✅' if exists else '❌ MISSING'} {os.path.basename(path)}")
    if not exists:
        print('     → Run 01_data_collection.ipynb first!')

## Step 2 — Load Raw Data

In [ ]:
laps_raw    = pd.read_csv(LAPS_RAW_PATH)
results_raw = pd.read_csv(RESULTS_RAW_PATH)

print('📊 Raw data loaded:')
print(f'   laps_raw:    {laps_raw.shape[0]:,} rows × {laps_raw.shape[1]} cols')
print(f'   results_raw: {results_raw.shape[0]:,} rows × {results_raw.shape[1]} cols')

print('\n📅 Laps per season:')
print(laps_raw.groupby('Season').size().rename('laps').to_string())

print('\n🔍 TrackStatus unique values (top 15):')
print(laps_raw['TrackStatus'].value_counts().head(15).to_string())

In [ ]:
laps_raw.head(3)

## Step 3 — Convert Timedelta Columns to Float Seconds

FastF1 stores times as strings like `'0 days 00:01:23.456000'`. We convert to plain float seconds.

In [ ]:
def to_seconds(series):
    """Convert timedelta strings or objects to float seconds. NaN on failure."""
    def _cvt(val):
        if pd.isna(val):
            return np.nan
        if isinstance(val, (int, float)):
            return float(val)
        try:
            return pd.Timedelta(val).total_seconds()
        except Exception:
            return np.nan
    return series.apply(_cvt)

laps = laps_raw.copy()

TIME_COLS = ['LapTime', 'Sector1Time', 'Sector2Time', 'Sector3Time', 'PitInTime', 'PitOutTime']

for col in TIME_COLS:
    if col in laps.columns:
        laps[col] = to_seconds(laps[col])

print('✅ Timedelta conversion done.')
print(f'   LapTime range: {laps["LapTime"].min():.1f}s – {laps["LapTime"].max():.1f}s')
print(f'   LapTime nulls: {laps["LapTime"].isna().sum():,}')

## Step 4 — Remove Pit-In and Pit-Out Laps

In [ ]:
before = len(laps)

pit_in  = laps['PitInTime'].notna()  if 'PitInTime'  in laps.columns else pd.Series(False, index=laps.index)
pit_out = laps['PitOutTime'].notna() if 'PitOutTime' in laps.columns else pd.Series(False, index=laps.index)

laps = laps[~pit_in & ~pit_out].copy()

print(f'🗑️  Pit-in/pit-out laps removed:  {before - len(laps):>7,}')
print(f'   ✅ Remaining:                  {len(laps):>7,}')

## Step 5 — Remove Safety Car / VSC / Red Flag Laps

TrackStatus is a compound number where digits encode flags:
- `4` = Safety Car, `5` = Red Flag, `6` = VSC, `7` = VSC ending  
We remove any lap where the status string contains 4, 5, or 6.

In [ ]:
before = len(laps)

if 'TrackStatus' in laps.columns:
    ts = laps['TrackStatus'].astype(str).str.strip()
    sc_mask = ts.str.contains('[456]', regex=True)
    laps = laps[~sc_mask].copy()

    print(f'🗑️  SC/VSC/Red flag laps removed:  {before - len(laps):>7,}')
    print(f'   ✅ Remaining:                   {len(laps):>7,}')
    print(f'\n   TrackStatus breakdown of kept laps:')
    print(laps['TrackStatus'].value_counts().head(10).to_string())
else:
    print('⚠️  TrackStatus column not found — skipping')

## Step 6 — Remove Missing or Impossible Lap Times

In [ ]:
before = len(laps)

laps = laps[laps['LapTime'].notna()].copy()
after_null = len(laps)

laps = laps[(laps['LapTime'] > 55) & (laps['LapTime'] < 200)].copy()

print(f'🗑️  Null LapTime removed:          {before - after_null:>7,}')
print(f'🗑️  Impossible LapTime removed:    {after_null - len(laps):>7,}  (< 55s or > 200s)')
print(f'   ✅ Remaining:                   {len(laps):>7,}')
print(f'\n   LapTime range now: {laps["LapTime"].min():.1f}s – {laps["LapTime"].max():.1f}s')

## Step 7 — Remove Outlier Laps (>3σ above race median)

In [ ]:
before = len(laps)

race_stats = (
    laps.groupby(['Season', 'RoundNumber'])['LapTime']
    .agg(race_median='median', race_std='std')
    .reset_index()
)

laps = laps.merge(race_stats, on=['Season', 'RoundNumber'], how='left')

upper_bound = laps['race_median'] + 3 * laps['race_std']
outlier_mask = laps['LapTime'] > upper_bound
laps = laps[~outlier_mask].copy()

print(f'🗑️  Outlier laps removed (>3σ):    {before - len(laps):>7,}')
print(f'   ✅ Remaining:                   {len(laps):>7,}')

## Step 8 — Drop DNF Drivers (≤50% laps completed)

In [ ]:
before = len(laps)

race_total_laps = (
    laps.groupby(['Season', 'RoundNumber'])['LapNumber']
    .max()
    .rename('total_race_laps')
    .reset_index()
)

driver_lap_counts = (
    laps.groupby(['Season', 'RoundNumber', 'Driver'])['LapNumber']
    .count()
    .rename('laps_completed')
    .reset_index()
)

driver_lap_counts = driver_lap_counts.merge(race_total_laps, on=['Season', 'RoundNumber'])
driver_lap_counts['completion_pct'] = driver_lap_counts['laps_completed'] / driver_lap_counts['total_race_laps']

valid_drivers = driver_lap_counts[driver_lap_counts['completion_pct'] > 0.50][['Season', 'RoundNumber', 'Driver']]
n_dropped = len(driver_lap_counts[driver_lap_counts['completion_pct'] <= 0.50])

laps = laps.merge(valid_drivers, on=['Season', 'RoundNumber', 'Driver'], how='inner')

print(f'🗑️  DNF driver-races dropped:       {n_dropped:>7,}  (completed ≤50% of laps)')
print(f'🗑️  Laps removed from DNFs:         {before - len(laps):>7,}')
print(f'   ✅ Remaining:                    {len(laps):>7,}')

## Step 9 — Normalize Lap Times Within Each Race

`LapTimeNorm = LapTime / race_median` — values < 1.0 are faster than median.

In [ ]:
race_medians = (
    laps.groupby(['Season', 'RoundNumber'])['LapTime']
    .median()
    .rename('race_median_final')
    .reset_index()
)
laps = laps.merge(race_medians, on=['Season', 'RoundNumber'], how='left')

laps['LapTimeNorm'] = laps['LapTime'] / laps['race_median_final']

for sec in ['Sector1Time', 'Sector2Time', 'Sector3Time']:
    if sec in laps.columns:
        sec_med = laps.groupby(['Season', 'RoundNumber'])[sec].transform('median')
        laps[f'{sec}Norm'] = laps[sec] / sec_med

print('✅ Lap times normalized.')
print(f'   LapTimeNorm — mean: {laps["LapTimeNorm"].mean():.4f} | std: {laps["LapTimeNorm"].std():.4f}')

## Step 10 — Clean Results Table

In [ ]:
results = results_raw.copy()

# Rename columns for consistency
results = results.rename(columns={
    'Abbreviation': 'Abbreviation',
    'GridPosition': 'GridPosition',
    'Position':     'Position',
})

# Convert Position columns to numeric
for col in ['GridPosition', 'Position']:
    if col in results.columns:
        results[col] = pd.to_numeric(results[col], errors='coerce')

# Compute position gain
# Pit lane starts are coded as GridPosition=0 — recode to 20
results['GridPosition'] = results['GridPosition'].fillna(20)
results.loc[results['GridPosition'] == 0, 'GridPosition'] = 20

results['PositionGain'] = results['GridPosition'] - results['Position']

# Drop rows with no finish position (DNS, DSQ, etc.)
results = results[results['Position'].notna()].copy()

print(f'✅ results_clean: {results.shape[0]:,} rows × {results.shape[1]} cols')
print(f'   PositionGain range: {results["PositionGain"].min():.0f} to {results["PositionGain"].max():.0f}')

## Step 11 — Save Clean Files

In [ ]:
laps.to_csv(LAPS_CLEAN_PATH, index=False)
results.to_csv(RESULTS_CLEAN_PATH, index=False)

print(f'💾 laps_clean.csv    → {laps.shape[0]:,} rows × {laps.shape[1]} cols')
print(f'💾 results_clean.csv → {results.shape[0]:,} rows × {results.shape[1]} cols')

# Summary
print('\n📊 Preprocessing summary:')
print(f'   Raw laps:   {len(laps_raw):,}')
print(f'   Clean laps: {len(laps):,}  ({len(laps)/len(laps_raw)*100:.1f}% retained)')